# Singapore Dengue + Weather Dataset Collection (2023 – Present)

**Purpose:** This notebook collects and constructs a dengue-weather dataset for Singapore
covering **2023-01-01 onwards**, continuing directly from the existing `singapore_dengue_weather_2013_2022.csv`.

**Data Sources:**
- 🦟 **Dengue Cases:** data.gov.sg / NEA Open API covers only through Dec 2022; 2023–2025 counts sourced manually from Singapore MOH/CDA Weekly Infectious Disease Bulletins (weekly cases → interpolated to daily)
- 🌤️ **Weather Data:** [Open-Meteo Historical API](https://open-meteo.com/) (free, no API key required)
  - Temperature (mean / max / min), Rainfall, Humidity, Wind Speed

**Output Columns (matching original dataset):**
`date, dengue_cases, temperature, temperature_max, temperature_min, rainfall, humidity, windspeed, windspeed_max, day_of_year, month, year, population_density, is_rainy_season, temperature_7d_avg, rainfall_7d_sum, humidity_7d_avg, cases_7d_avg`

---
**Author:** Ong Shi Kai  
**FYP Title:** A Hybrid Deep Learning Approach for Dengue Fever Prediction in real-world case study  
**Institution:** TAR UMT, Pulau Penang

## Cell 1 — Install & Import Libraries

In [1]:
# Install required libraries (run once)
!pip install requests pandas numpy openmeteo-requests requests-cache retry-requests openpyxl xlrd

import requests
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For Open-Meteo
import openmeteo_requests
import requests_cache
from retry_requests import retry

print('✅ All libraries imported successfully.')

Defaulting to user installation because normal site-packages is not writeable
✅ All libraries imported successfully.


## Cell 2 — Configuration & Parameters

In [2]:
# ─── CONFIGURATION ───────────────────────────────────────────────

# Date range: continue from 2023-01-01
START_DATE = '2023-01-01'
END_DATE   = '2025-12-31'   # Adjust to today or desired end date

# Singapore coordinates (for weather API)
LATITUDE  = 1.3521
LONGITUDE = 103.8198

# Singapore population density (persons/km²) — approximate recent figure
# Source: Singapore Department of Statistics
POPULATION_DENSITY = 8041.0  # 2023 estimate

# Rainy season months for Singapore (Northeast Monsoon: Nov–Jan, Southwest Monsoon: May–Jul)
RAINY_MONTHS = [11, 12, 1, 5, 6, 7]

print(f'📅 Data collection period: {START_DATE} → {END_DATE}')
print(f'📍 Location: Singapore ({LATITUDE}°N, {LONGITUDE}°E)')

📅 Data collection period: 2023-01-01 → 2025-12-31
📍 Location: Singapore (1.3521°N, 103.8198°E)


## Cell 3 — Fetch Dengue Cases from Singapore data.gov.sg API

In [3]:
def fetch_dengue_cases_datagov():
    """
    Fetch weekly dengue cases from Singapore data.gov.sg.
    Dataset ID: d_c3942c2a4b580f61b576b7c44c4f5c8c
    Source: https://data.gov.sg/datasets/d_c3942c2a4b580f61b576b7c44c4f5c8c/view
    """
    print('🔄 Fetching dengue data from data.gov.sg ...')
    
    # data.gov.sg API endpoint for weekly dengue cases
    url = 'https://data.gov.sg/api/action/datastore_search'
    dataset_id = 'd_c3942c2a4b580f61b576b7c44c4f5c8c'
    
    all_records = []
    offset = 0
    limit = 100
    
    while True:
        params = {
            'resource_id': dataset_id,
            'limit': limit,
            'offset': offset
        }
        try:
            response = requests.get(url, params=params, timeout=30)
            data = response.json()
            records = data['result']['records']
            if not records:
                break
            all_records.extend(records)
            offset += limit
            if offset >= data['result']['total']:
                break
        except Exception as e:
            print(f'⚠️  API error at offset {offset}: {e}')
            break
    
    if all_records:
        df = pd.DataFrame(all_records)
        print(f'✅ Retrieved {len(df)} records from data.gov.sg')
        return df
    else:
        print('⚠️  No records returned. Trying alternative endpoint...')
        return None


def fetch_dengue_alternative():
    """
    Alternative: Fetch from NEA weekly dengue data via data.gov.sg datasets API.
    """
    print('🔄 Trying alternative dengue data endpoint...')
    
    # Try the weekly infectious diseases dataset
    url = 'https://data.gov.sg/api/action/datastore_search'
    # Weekly Infectious Disease Bulletin dataset
    dataset_id = 'd_13c7b9da3fdbff6e7bc7f0f8f7c09f0e'
    
    params = {
        'resource_id': dataset_id,
        'limit': 500,
        'filters': json.dumps({'disease': 'Dengue Fever'})
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        data = response.json()
        if data.get('success') and data['result']['records']:
            df = pd.DataFrame(data['result']['records'])
            print(f'✅ Retrieved {len(df)} records from alternative endpoint')
            return df
    except Exception as e:
        print(f'⚠️  Alternative endpoint error: {e}')
    
    return None


# Run the fetch
dengue_raw = fetch_dengue_cases_datagov()
if dengue_raw is None:
    dengue_raw = fetch_dengue_alternative()

if dengue_raw is not None:
    print('\n📋 Raw dengue data preview:')
    print(dengue_raw.head(10))
    print(f'\nColumns: {list(dengue_raw.columns)}')
else:
    print('\n⚠️  Automatic fetch failed. See Cell 4 for manual input instructions.')

🔄 Fetching dengue data from data.gov.sg ...
⚠️  API error at offset 0: Extra data: line 1 column 5 (char 4)
⚠️  No records returned. Trying alternative endpoint...
🔄 Trying alternative dengue data endpoint...

⚠️  Automatic fetch failed. See Cell 4 for manual input instructions.


## Cell 4 — Manual Dengue Data Input (Fallback if API fails)

> **Instructions if Cell 3 fails:**
> 1. Visit: https://data.gov.sg/datasets?query=dengue
> 2. Download **"Weekly Infectious Disease Bulletin"** CSV
> 3. Filter rows where disease = 'Dengue Fever'
> 4. Replace the `manual_data` dictionary below with the actual data
> 5. Or visit: https://www.nea.gov.sg/dengue-zika/dengue/dengue-cases

In [4]:
import glob, os

# ════════════════════════════════════════════════════════════════════
# CDA (Communicable Diseases Agency) Weekly Bulletin Excel Files
# Download from:
#   2023: https://www.cda.gov.sg/resources/weekly-infectious-diseases-bulletin-2023/
#   2024: https://www.cda.gov.sg/resources/weekly-infectious-diseases-bulletin-2024/
#   2025: https://www.cda.gov.sg/resources/weekly-infectious-diseases-bulletin-2025/
# Save as: moh_2023.xlsx (or .xls), moh_2024.xlsx, moh_2025.xlsx
# Place in the SAME FOLDER as this notebook, then run this cell.
# ════════════════════════════════════════════════════════════════════

# ── CDA FILE FORMAT (IMPORTANT) ──────────────────────────────────────
# Row 0: top-level headers (Epidemiology Wk | Start to End | ... | Total Number ...)
# Row 1: disease names   (col 3=Cholera ... col 11=Dengue ... col 45=HFMD)
# Row 2+: weekly data
# Col 0 = week number, Col 1 = date range (e.g. "1/1/2023 - 7/1/2023")
# Col 11 = Dengue weekly case count  <-- THIS IS THE TARGET COLUMN
# NOTE: 2023 file may have .xls extension but is actually xlsx format
#        Always use engine="openpyxl" for all CDA files.
# ─────────────────────────────────────────────────────────────────────

def load_cda_excel_files():
    """Load all moh_*.xlsx / moh_*.xls files using openpyxl."""
    try:
        import openpyxl
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "--quiet"])

    xlsx_files = sorted(glob.glob("moh_*.xlsx") + glob.glob("moh_*.xls"))
    if not xlsx_files:
        print("No moh_*.xlsx or moh_*.xls files found. Using MANUAL_DATA fallback.")
        return None

    frames = []
    for fp in xlsx_files:
        try:
            # Always use openpyxl — CDA .xls files are actually xlsx format
            df = pd.read_excel(fp, sheet_name=0, engine="openpyxl", header=None)
            # Skip rows 0 and 1 (headers), keep data rows
            data   = df.iloc[2:].copy().reset_index(drop=True)
            weeks  = pd.to_numeric(data[0], errors="coerce")
            dates  = data[1].astype(str)
            dengue = pd.to_numeric(data[11], errors="coerce")  # col 11 = Dengue
            mask   = weeks.notna() & dengue.notna()
            chunk  = pd.DataFrame({
                "week_num":     weeks[mask].astype(int).values,
                "date_range":   dates[mask].values,
                "dengue_cases": dengue[mask].astype(int).values,
            })
            frames.append(chunk)
            print(f"   Loaded {fp}: {len(chunk)} weeks | dengue total = {chunk['dengue_cases'].sum()}")
        except Exception as e:
            print(f"   Failed {fp}: {e}")

    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    print(f"Combined: {len(combined)} weekly rows")
    return combined


def date_range_to_start(date_range_str):
    """
    Parse CDA date range strings to the start (Monday) date.
    Handles formats: "1/1/2023 - 7/1/2023" or "29/12/2024 - 04/01/2025"
    """
    try:
        start_str = str(date_range_str).split(" - ")[0].strip()
        return datetime.strptime(start_str, "%d/%m/%Y")
    except Exception:
        return pd.NaT


def build_dengue_daily_from_cda(df_weekly_raw):
    """
    Convert CDA weekly dengue counts to a daily series.
    - Uses date_range col to extract the week start date
    - Divides weekly count by 7 for each day
    - Anchor row ensures 2023-01-01 is not left as zero
    """
    df = df_weekly_raw.copy()
    df["week_date"] = df["date_range"].apply(date_range_to_start)
    df = df.dropna(subset=["week_date"]).sort_values("week_date").reset_index(drop=True)

    # Filter to our target date range
    df = df[df["week_date"] >= pd.Timestamp(START_DATE)].copy()
    print(f"   Weeks after filtering to {START_DATE}: {len(df)}")

    # Anchor: ensure START_DATE has a value
    first_val = float(df.iloc[0]["dengue_cases"]) if len(df) > 0 else 0.0
    anchor = pd.DataFrame({"week_date": [pd.Timestamp(START_DATE)],
                           "dengue_cases": [first_val]})
    df_w = pd.concat([anchor, df[["week_date","dengue_cases"]]], ignore_index=True)
    df_w = df_w.drop_duplicates("week_date").sort_values("week_date").reset_index(drop=True)
    df_w["dengue_cases"] = df_w["dengue_cases"].astype(float)

    date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
    daily_cases = []
    for d in date_range:
        mask = df_w["week_date"] <= d
        val  = float(df_w[mask].iloc[-1]["dengue_cases"]) if mask.any() else 0.0
        daily_cases.append(round(val / 7, 4))

    df_daily = pd.DataFrame({"date": date_range, "dengue_cases": daily_cases})
    return df_daily


# ── Run ──────────────────────────────────────────────────────────────
df_cda = load_cda_excel_files()

if df_cda is not None:
    try:
        dengue_daily = build_dengue_daily_from_cda(df_cda)
        print("Built dengue_daily from CDA Excel files.")
    except Exception as e:
        print(f"Error processing CDA files: {e}")
        import traceback; traceback.print_exc()
        dengue_daily = None
else:
    dengue_daily = None

if dengue_daily is None:
    print("CDA files unavailable. No MANUAL_DATA fallback — please provide the CDA Excel files.")
    date_range   = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
    dengue_daily = pd.DataFrame({"date": date_range, "dengue_cases": np.nan})

print()
print(f"Daily dengue records : {len(dengue_daily)}")
print(f"Date range           : {dengue_daily['date'].min().date()} -> {dengue_daily['date'].max().date()}")
print(f"dengue_cases min     : {dengue_daily['dengue_cases'].min():.4f}")
print(f"dengue_cases max     : {dengue_daily['dengue_cases'].max():.4f}")
print(f"dengue_cases mean    : {dengue_daily['dengue_cases'].mean():.4f}")
print(f"NaN count            : {dengue_daily['dengue_cases'].isna().sum()}")
dengue_daily.head(10)


   Loaded moh_2023.xlsx.xls: 52 weeks | dengue total = 9938
   Loaded moh_2024.xlsx.xlsx: 52 weeks | dengue total = 13606
   Loaded moh_2025.xlsx.xlsx: 53 weeks | dengue total = 4017
Combined: 157 weekly rows
   Weeks after filtering to 2023-01-01: 157
Built dengue_daily from CDA Excel files.

Daily dengue records : 1096
Date range           : 2023-01-01 -> 2025-12-31
dengue_cases min     : 3.8571
dengue_cases max     : 73.4286
dengue_cases mean    : 25.1320
NaN count            : 0


,date,dengue_cases
0,2023-01-01,39.8571
1,2023-01-02,39.8571
2,2023-01-03,39.8571
3,2023-01-04,39.8571
4,2023-01-05,39.8571
5,2023-01-06,39.8571
6,2023-01-07,39.8571
7,2023-01-08,39.7143
8,2023-01-09,39.7143
9,2023-01-10,39.7143


## Cell 5 — Fetch Weather Data from Open-Meteo API

In [5]:
def fetch_weather_openmeteo(start_date, end_date, lat, lon):
    """
    Fetch daily weather data for Singapore from Open-Meteo Historical API.
    
    Variables fetched:
    - temperature_2m_mean      → temperature
    - temperature_2m_max       → temperature_max
    - temperature_2m_min       → temperature_min
    - precipitation_sum        → rainfall
    - relative_humidity_2m_mean→ humidity
    - wind_speed_10m_mean      → windspeed
    - wind_speed_10m_max       → windspeed_max
    
    Source: https://open-meteo.com/en/docs/historical-weather-api
    """
    print('🔄 Fetching weather data from Open-Meteo Historical API...')
    
    # Setup Open-Meteo API client with cache
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date':   end_date,
        'daily': [
            'temperature_2m_mean',
            'temperature_2m_max',
            'temperature_2m_min',
            'precipitation_sum',
            'relative_humidity_2m_mean',
            'wind_speed_10m_mean',
            'wind_speed_10m_max'
        ],
        'timezone': 'Asia/Singapore'
    }
    
    responses = openmeteo.weather_api(url, params=params)
    response  = responses[0]
    
    print(f'✅ Weather data fetched for: {response.Latitude()}°N, {response.Longitude()}°E')
    print(f'   Timezone: {response.Timezone()} (UTC offset: {response.UtcOffsetSeconds()/3600}h)')
    
    # Parse daily data
    daily = response.Daily()
    dates = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit='s', utc=True).tz_localize(None),
        end=pd.to_datetime(daily.TimeEnd(), unit='s', utc=True).tz_localize(None),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive='left'
    )
    
    df_weather = pd.DataFrame({
        'date':             dates,
        'temperature':      daily.Variables(0).ValuesAsNumpy(),
        'temperature_max':  daily.Variables(1).ValuesAsNumpy(),
        'temperature_min':  daily.Variables(2).ValuesAsNumpy(),
        'rainfall':         daily.Variables(3).ValuesAsNumpy(),
        'humidity':         daily.Variables(4).ValuesAsNumpy(),
        'windspeed':        daily.Variables(5).ValuesAsNumpy(),
        'windspeed_max':    daily.Variables(6).ValuesAsNumpy(),
    })
    
    # Round to 2 decimal places
    weather_cols = ['temperature','temperature_max','temperature_min','rainfall','humidity','windspeed','windspeed_max']
    df_weather[weather_cols] = df_weather[weather_cols].round(2)
    
    print(f'✅ Weather records: {len(df_weather)} days ({df_weather["date"].min().date()} → {df_weather["date"].max().date()})')
    return df_weather


# Fetch weather data
df_weather = fetch_weather_openmeteo(START_DATE, END_DATE, LATITUDE, LONGITUDE)
print('\n📋 Weather data preview:')
df_weather.head()

🔄 Fetching weather data from Open-Meteo Historical API...
✅ Weather data fetched for: 1.3708258867263794°N, 103.8023681640625°E
   Timezone: b'Asia/Singapore' (UTC offset: 8.0h)
✅ Weather records: 1096 days (2022-12-31 → 2025-12-30)

📋 Weather data preview:


,date,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max
0,2022-12-31 16:00:00,25.790001,28.90,23.400000,0.1,81.610001,11.21,16.389999
1,2023-01-01 16:00:00,26.190001,30.25,22.950001,1.1,81.209999,11.32,21.080000
2,2023-01-02 16:00:00,25.730000,29.90,23.549999,8.5,86.660004,9.65,16.870001
3,2023-01-03 16:00:00,25.690001,28.85,23.450001,1.1,84.339996,13.81,20.959999
4,2023-01-04 16:00:00,25.809999,30.00,23.549999,4.6,86.180000,9.56,15.510000


## Cell 6 — Merge Dengue + Weather Data

In [6]:
# Ensure consistent date format
dengue_daily["date"] = pd.to_datetime(dengue_daily["date"]).dt.normalize()
df_weather["date"]   = pd.to_datetime(df_weather["date"]).dt.normalize()

# LEFT merge — keeps all dengue dates including 2025-12-31.
# Open-Meteo weather ends 2025-12-30; inner merge silently drops that last day.
df = pd.merge(dengue_daily, df_weather, on="date", how="left")
df = df.sort_values("date").reset_index(drop=True)

# Forward-fill only the 7 raw measurement columns that actually exist after merge.
# day_of_year / month / year / population_density / is_rainy_season
# are NOT here yet — they are computed in Cell 7 (add_derived_features).
raw_weather_cols = [
    "temperature", "temperature_max", "temperature_min",
    "rainfall", "humidity", "windspeed", "windspeed_max"
]
df[raw_weather_cols] = df[raw_weather_cols].ffill()

print(f"Merged dataset : {len(df)} records")
print(f"Date range     : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Weather NaN    : {df[raw_weather_cols].isna().sum().sum()}")
print(f"Columns        : {list(df.columns)}")
df.head()


Merged dataset : 1096 records
Date range     : 2023-01-01 -> 2025-12-31
Weather NaN    : 0
Columns        : ['date', 'dengue_cases', 'temperature', 'temperature_max', 'temperature_min', 'rainfall', 'humidity', 'windspeed', 'windspeed_max']


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max
0,2023-01-01,39.8571,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000
1,2023-01-02,39.8571,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001
2,2023-01-03,39.8571,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999
3,2023-01-04,39.8571,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000
4,2023-01-05,39.8571,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000


## Cell 7 — Add Derived Features (matching original dataset columns)

In [7]:
def add_derived_features(df):
    """
    Add derived columns to match the original 2013-2022 dataset schema.
    Rolling features are recomputed fresh here to avoid any stale values.
    cases_7d_avg uses 4dp to match dengue_cases precision.
    """
    df = df.copy().sort_values("date").reset_index(drop=True)

    # Calendar features (recomputed from date to ensure correctness)
    df["day_of_year"]        = df["date"].dt.dayofyear
    df["month"]              = df["date"].dt.month
    df["year"]               = df["date"].dt.year

    # Static features
    df["population_density"] = POPULATION_DENSITY
    df["is_rainy_season"]    = df["month"].apply(lambda m: 1 if m in RAINY_MONTHS else 0)

    # 7-day rolling features — recomputed fresh
    df["temperature_7d_avg"] = df["temperature"].rolling(window=7, min_periods=1).mean().round(2)
    df["rainfall_7d_sum"]    = df["rainfall"].rolling(window=7, min_periods=1).sum().round(2)
    df["humidity_7d_avg"]    = df["humidity"].rolling(window=7, min_periods=1).mean().round(2)
    # Round to 4dp to match dengue_cases precision (not 2dp)
    df["cases_7d_avg"]       = df["dengue_cases"].rolling(window=7, min_periods=1).mean().round(4)

    # Format date as string
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    print("Derived features added.")
    return df


df = add_derived_features(df)

COLUMN_ORDER = [
    "date", "dengue_cases", "temperature", "temperature_max", "temperature_min",
    "rainfall", "humidity", "windspeed", "windspeed_max",
    "day_of_year", "month", "year", "population_density", "is_rainy_season",
    "temperature_7d_avg", "rainfall_7d_sum", "humidity_7d_avg", "cases_7d_avg"
]
df = df[COLUMN_ORDER]
print(f"Final shape: {df.shape}")
print(f"Columns   : {list(df.columns)}")
df.head(5)


Derived features added.
Final shape: (1096, 18)
Columns   : ['date', 'dengue_cases', 'temperature', 'temperature_max', 'temperature_min', 'rainfall', 'humidity', 'windspeed', 'windspeed_max', 'day_of_year', 'month', 'year', 'population_density', 'is_rainy_season', 'temperature_7d_avg', 'rainfall_7d_sum', 'humidity_7d_avg', 'cases_7d_avg']


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
0,2023-01-01,39.8571,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000,1,1,2023,8041.0,1,26.19,1.1,81.21,39.8571
1,2023-01-02,39.8571,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001,2,1,2023,8041.0,1,25.96,9.6,83.94,39.8571
2,2023-01-03,39.8571,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999,3,1,2023,8041.0,1,25.87,10.7,84.07,39.8571
3,2023-01-04,39.8571,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000,4,1,2023,8041.0,1,25.86,15.3,84.60,39.8571
4,2023-01-05,39.8571,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000,5,1,2023,8041.0,1,25.90,17.7,84.79,39.8571


## Cell 8 — Data Quality Check

In [8]:
print('=' * 55)
print('📊 DATA QUALITY REPORT')
print('=' * 55)
print(f'\nTotal records   : {len(df)}')
print(f'Date range      : {df["date"].min()} → {df["date"].max()}')
print(f'\n📌 Missing values per column:')
missing = df.isnull().sum()
for col, count in missing.items():
    status = '⚠️ ' if count > 0 else '✅'
    print(f'   {status} {col:25s}: {count}')

print(f'\n📌 Basic statistics:')
print(df[['dengue_cases','temperature','rainfall','humidity','windspeed']].describe().round(2))

📊 DATA QUALITY REPORT

Total records   : 1096
Date range      : 2023-01-01 → 2025-12-31

📌 Missing values per column:
   ✅ date                     : 0
   ✅ dengue_cases             : 0
   ✅ temperature              : 0
   ✅ temperature_max          : 0
   ✅ temperature_min          : 0
   ✅ rainfall                 : 0
   ✅ humidity                 : 0
   ✅ windspeed                : 0
   ✅ windspeed_max            : 0
   ✅ day_of_year              : 0
   ✅ month                    : 0
   ✅ year                     : 0
   ✅ population_density       : 0
   ✅ is_rainy_season          : 0
   ✅ temperature_7d_avg       : 0
   ✅ rainfall_7d_sum          : 0
   ✅ humidity_7d_avg          : 0
   ✅ cases_7d_avg             : 0

📌 Basic statistics:
       dengue_cases  temperature  rainfall  humidity  windspeed
count       1096.00      1096.00   1096.00   1096.00    1096.00
mean          25.13        26.72      9.24     85.64       7.39
std           15.59         1.01     10.82      4.46     

## Cell 9 — Handle Missing Values

In [9]:
# Fill any remaining NaN values using linear interpolation (same method as original dataset)
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Interpolate numeric columns
df_clean[numeric_cols] = df_clean[numeric_cols].interpolate(method='linear', limit_direction='both')

# Forward fill any remaining NaN (edge cases)
df_clean[numeric_cols] = df_clean[numeric_cols].ffill().bfill()

missing_after = df_clean.isnull().sum().sum()
print(f'✅ Missing values after imputation: {missing_after}')

# Recompute rolling features after imputation
df_clean['temperature_7d_avg'] = df_clean['temperature'].rolling(window=7, min_periods=1).mean().round(2)
df_clean['rainfall_7d_sum']    = df_clean['rainfall'].rolling(window=7, min_periods=1).sum().round(2)
df_clean['humidity_7d_avg']    = df_clean['humidity'].rolling(window=7, min_periods=1).mean().round(2)
df_clean['cases_7d_avg']       = df_clean['dengue_cases'].rolling(window=7, min_periods=1).mean().round(4)

print('✅ Rolling features recomputed after imputation.')
df_clean.tail()

✅ Missing values after imputation: 0
✅ Rolling features recomputed after imputation.


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
1091,2025-12-27,4.5714,25.559999,30.750000,22.750000,15.9,89.489998,6.07,10.12,361,12,2025,8041.0,1,26.61,45.0,83.97,4.5714
1092,2025-12-28,5.4286,25.600000,29.450001,22.900000,10.7,88.150002,9.04,14.47,362,12,2025,8041.0,1,26.38,55.7,85.60,4.6939
1093,2025-12-29,5.4286,26.559999,30.850000,23.350000,0.2,78.349998,8.97,15.88,363,12,2025,8041.0,1,26.31,54.9,85.24,4.8163
1094,2025-12-30,5.4286,26.420000,30.400000,23.549999,0.5,79.660004,8.47,12.24,364,12,2025,8041.0,1,26.18,54.8,85.17,4.9388
1095,2025-12-31,5.4286,26.420000,30.400000,23.549999,0.5,79.660004,8.47,12.24,365,12,2025,8041.0,1,26.10,52.7,84.61,5.0612


## Cell 10 — Save Output CSV

In [10]:
output_filename = f'singapore_dengue_weather_{START_DATE[:4]}_{END_DATE[:4]}.csv'

df_clean.to_csv(output_filename, index=False)

print(f'✅ Dataset saved as: {output_filename}')
print(f'   Shape : {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'   Size  : ~{df_clean.memory_usage(deep=True).sum() / 1024:.1f} KB in memory')
print(f'\n📋 Final preview:')
df_clean.head()

✅ Dataset saved as: singapore_dengue_weather_2023_2025.csv
   Shape : 1096 rows × 18 columns
   Size  : ~166.0 KB in memory

📋 Final preview:


,date,dengue_cases,temperature,temperature_max,temperature_min,rainfall,humidity,windspeed,windspeed_max,day_of_year,month,year,population_density,is_rainy_season,temperature_7d_avg,rainfall_7d_sum,humidity_7d_avg,cases_7d_avg
0,2023-01-01,39.8571,26.190001,30.250000,22.950001,1.1,81.209999,11.32,21.080000,1,1,2023,8041.0,1,26.19,1.1,81.21,39.8571
1,2023-01-02,39.8571,25.730000,29.900000,23.549999,8.5,86.660004,9.65,16.870001,2,1,2023,8041.0,1,25.96,9.6,83.94,39.8571
2,2023-01-03,39.8571,25.690001,28.850000,23.450001,1.1,84.339996,13.81,20.959999,3,1,2023,8041.0,1,25.87,10.7,84.07,39.8571
3,2023-01-04,39.8571,25.809999,30.000000,23.549999,4.6,86.180000,9.56,15.510000,4,1,2023,8041.0,1,25.86,15.3,84.60,39.8571
4,2023-01-05,39.8571,26.100000,29.200001,23.900000,2.4,85.550003,9.65,15.140000,5,1,2023,8041.0,1,25.90,17.7,84.79,39.8571


## Cell 11 — (Optional) Combine with Original 2013–2022 Dataset

In [11]:
# ⚠️  Only run this cell if you want to MERGE with the original dataset
# Update the path below to match where your original file is located

ORIGINAL_FILE = 'singapore_dengue_weather_2013_2022.csv'   # ← Update path if needed

try:
    df_original = pd.read_csv(ORIGINAL_FILE)
    print(f'✅ Original dataset loaded: {df_original.shape}')
    
    df_combined = pd.concat([df_original, df_clean], ignore_index=True)
    df_combined = df_combined.sort_values('date').reset_index(drop=True)
    df_combined = df_combined.drop_duplicates(subset=['date'])
    
    combined_filename = 'singapore_dengue_weather_2013_2025_combined.csv'
    df_combined.to_csv(combined_filename, index=False)
    
    print(f'✅ Combined dataset saved: {combined_filename}')
    print(f'   Shape : {df_combined.shape[0]} rows × {df_combined.shape[1]} columns')
    print(f'   Range : {df_combined["date"].min()} → {df_combined["date"].max()}')

except FileNotFoundError:
    print(f'⚠️  Original file "{ORIGINAL_FILE}" not found.')
    print('    Upload it to the same folder and re-run this cell.')

✅ Original dataset loaded: (3652, 18)
✅ Combined dataset saved: singapore_dengue_weather_2013_2025_combined.csv
   Shape : 4748 rows × 18 columns
   Range : 2013-01-01 → 2025-12-31


---
## 📌 Notes for Chapter 4 Documentation

| Item | Detail |
|------|--------|
| **Dengue Source** | data.gov.sg/NEA Open API (≤2022 only); 2023–2025 from MOH/CDA Weekly Infectious Disease Bulletins (manual) |
| **Weather Source** | [Open-Meteo Historical Weather API](https://open-meteo.com/) — ERA5 reanalysis |
| **Date Coverage** | 2023-01-01 to 2025-12-31 (extending original 2013–2022 dataset) |
| **Granularity** | Daily records |
| **Dengue Aggregation** | Weekly NEA case counts divided by 7 for daily interpolation |
| **Missing Values** | Linear interpolation (consistent with original dataset pipeline) |
| **Rolling Features** | 7-day window for temperature avg, rainfall sum, humidity avg, cases avg |
| **Rainy Season** | Binary flag for Nov–Jan (NE Monsoon) and May–Jul (SW Monsoon) |

In [12]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# ── 换成你的实际文件路径 ──────────────────────────────────────
df = pd.read_csv('singapore_dengue_weather_2013_2022.csv', parse_dates=['date'])
# ─────────────────────────────────────────────────────────────

series = df['dengue_cases'].dropna()

# (1) Raw dengue_cases
r1 = adfuller(series)
print('=== Raw dengue_cases ===')
print(f'  ADF Statistic : {r1[0]:.4f}')
print(f'  p-value       : {r1[1]:.6f}')
print(f'  Critical Values:')
for k, v in r1[4].items():
    print(f'    {k}: {v:.4f}')

# (2) First difference (lag 1)
r2 = adfuller(series.diff().dropna())
print('\n=== First difference (lag 1) ===')
print(f'  ADF Statistic : {r2[0]:.4f}')
print(f'  p-value       : {r2[1]:.6f}')
for k, v in r2[4].items():
    print(f'    {k}: {v:.4f}')

# (3) Seasonal difference (lag 7)
r3 = adfuller(series.diff(7).dropna())
print('\n=== Seasonal difference (lag 7) ===')
print(f'  ADF Statistic : {r3[0]:.4f}')
print(f'  p-value       : {r3[1]:.6f}')
for k, v in r3[4].items():
    print(f'    {k}: {v:.4f}')

=== Raw dengue_cases ===
  ADF Statistic : -4.6911
  p-value       : 0.000087
  Critical Values:
    1%: -3.4322
    5%: -2.8623
    10%: -2.5672

=== First difference (lag 1) ===
  ADF Statistic : -7.1036
  p-value       : 0.000000
    1%: -3.4322
    5%: -2.8623
    10%: -2.5672

=== Seasonal difference (lag 7) ===
  ADF Statistic : -6.7329
  p-value       : 0.000000
    1%: -3.4322
    5%: -2.8623
    10%: -2.5672


In [13]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller

# ── 换成你的实际文件路径 ──────────────────────────────────────
df = pd.read_csv('singapore_dengue_weather_2023_2025.csv', parse_dates=['date'])
# ─────────────────────────────────────────────────────────────

series = df['dengue_cases'].dropna()

# (1) Raw dengue_cases
r1 = adfuller(series)
print('=== Raw dengue_cases ===')
print(f'  ADF Statistic : {r1[0]:.4f}')
print(f'  p-value       : {r1[1]:.6f}')
print(f'  Critical Values:')
for k, v in r1[4].items():
    print(f'    {k}: {v:.4f}')

# (2) First difference (lag 1)
r2 = adfuller(series.diff().dropna())
print('\n=== First difference (lag 1) ===')
print(f'  ADF Statistic : {r2[0]:.4f}')
print(f'  p-value       : {r2[1]:.6f}')
for k, v in r2[4].items():
    print(f'    {k}: {v:.4f}')

# (3) Seasonal difference (lag 7)
r3 = adfuller(series.diff(7).dropna())
print('\n=== Seasonal difference (lag 7) ===')
print(f'  ADF Statistic : {r3[0]:.4f}')
print(f'  p-value       : {r3[1]:.6f}')
for k, v in r3[4].items():
    print(f'    {k}: {v:.4f}')

=== Raw dengue_cases ===
  ADF Statistic : -1.8730
  p-value       : 0.344849
  Critical Values:
    1%: -3.4365
    5%: -2.8642
    10%: -2.5682

=== First difference (lag 1) ===
  ADF Statistic : -5.7984
  p-value       : 0.000000
    1%: -3.4365
    5%: -2.8642
    10%: -2.5682

=== Seasonal difference (lag 7) ===
  ADF Statistic : -4.7681
  p-value       : 0.000063
    1%: -3.4365
    5%: -2.8643
    10%: -2.5682


In [14]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

# ── Load Dataset ──────────────────────────────────────────────
df = pd.read_csv('singapore_dengue_weather_2013_2022.csv', parse_dates=['date'])

# ═══════════════════════════════════════════════════════════════
# SECTION 1: DATA QUALITY REPORT
# ═══════════════════════════════════════════════════════════════
print('=' * 60)
print('📊 DATA QUALITY REPORT — 2013–2022 Benchmark Dataset')
print('=' * 60)

print(f'\nTotal records   : {len(df)}')
print(f'Date range      : {df["date"].min().date()} → {df["date"].max().date()}')

# Date continuity check
expected_days = (df['date'].max() - df['date'].min()).days + 1
print(f'Expected days   : {expected_days}')
print(f'Gaps detected   : {expected_days - len(df)}')

print(f'\n📌 Missing values per column:')
missing = df.isnull().sum()
for col, count in missing.items():
    status = '⚠️ ' if count > 0 else '✅'
    print(f'   {status} {col:30s}: {count}')

# ═══════════════════════════════════════════════════════════════
# SECTION 2: DESCRIPTIVE STATISTICS — dengue_cases
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('📊 DESCRIPTIVE STATISTICS — dengue_cases')
print('=' * 60)

s = df['dengue_cases']
print(f'  Count              : {s.count()}')
print(f'  Mean               : {s.mean():.1f} cases/day')
print(f'  Median             : {s.median():.1f} cases/day')
print(f'  Std Dev            : {s.std():.1f}')
print(f'  Min                : {s.min():.1f} cases/day')
print(f'  Max                : {s.max():.1f} cases/day')
print(f'  25th Percentile    : {s.quantile(0.25):.1f}')
print(f'  75th Percentile    : {s.quantile(0.75):.1f}')
print(f'  90th Percentile    : {s.quantile(0.90):.1f}')
print(f'  Skewness           : {s.skew():.2f}')

print('\n📌 Annual Std Dev (heteroscedasticity check):')
annual_std = df.groupby('year')['dengue_cases'].std().round(1)
for yr, val in annual_std.items():
    print(f'   {yr}: {val}')

# ═══════════════════════════════════════════════════════════════
# SECTION 3: DESCRIPTIVE STATISTICS — Meteorological Variables
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('📊 DESCRIPTIVE STATISTICS — Meteorological Variables')
print('=' * 60)

met_cols = ['temperature', 'rainfall', 'humidity', 'windspeed']
met_stats = df[met_cols].agg(['mean','std','min','max']).T
met_stats['skewness'] = df[met_cols].skew()
print(met_stats.round(2).to_string())

# ═══════════════════════════════════════════════════════════════
# SECTION 4: ADF STATIONARITY TEST
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('📊 ADF STATIONARITY TEST — dengue_cases')
print('=' * 60)

series = df['dengue_cases'].dropna()

tests = [
    ('Raw dengue_cases',          series),
    ('First difference (lag 1)',  series.diff().dropna()),
    ('Seasonal difference (lag 7)', series.diff(7).dropna()),
]

for label, s in tests:
    r = adfuller(s)
    print(f'\n  {label}')
    print(f'    ADF Statistic : {r[0]:.4f}')
    print(f'    p-value       : {r[1]:.6f}')
    print(f'    Stationary    : {"Yes ✅" if r[1] < 0.05 else "No ❌"}')
    print(f'    Critical Values:')
    for k, v in r[4].items():
        print(f'      {k}: {v:.4f}')

print('\n' + '=' * 60)
print('✅ All checks complete.')
print('=' * 60)

📊 DATA QUALITY REPORT — 2013–2022 Benchmark Dataset

Total records   : 3652
Date range      : 2013-01-01 → 2022-12-31
Expected days   : 3652
Gaps detected   : 0

📌 Missing values per column:
   ✅ date                          : 0
   ✅ dengue_cases                  : 0
   ✅ temperature                   : 0
   ✅ temperature_max               : 0
   ✅ temperature_min               : 0
   ✅ rainfall                      : 0
   ✅ humidity                      : 0
   ✅ windspeed                     : 0
   ✅ windspeed_max                 : 0
   ✅ day_of_year                   : 0
   ✅ month                         : 0
   ✅ year                          : 0
   ✅ population_density            : 0
   ✅ is_rainy_season               : 0
   ✅ temperature_7d_avg            : 0
   ✅ rainfall_7d_sum               : 0
   ✅ humidity_7d_avg               : 0
   ✅ cases_7d_avg                  : 0

📊 DESCRIPTIVE STATISTICS — dengue_cases
  Count              : 3652
  Mean               : 43.8 cases/day
